# Scania APS Failure Prediction — EDA


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path(".").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Imports OK")


Imports OK


In [2]:
SCANIA_DIR = PROJECT_DIR / "Datasets" / "SCANIA" / "SCANIA"

TRAIN_OPS  = SCANIA_DIR / "train_operational_readouts.csv"
TRAIN_TTE  = SCANIA_DIR / "train_tte.csv"
VAL_OPS    = SCANIA_DIR / "validation_operational_readouts.csv"
VAL_LABELS = SCANIA_DIR / "validation_labels.csv"
TEST_OPS   = SCANIA_DIR / "test_operational_readouts.csv"

print(f"Train ops : {TRAIN_OPS.exists()}")
print(f"Val ops   : {VAL_OPS.exists()}")
print(f"Test ops  : {TEST_OPS.exists()}")


Train ops : True
Val ops   : True
Test ops  : True


---


In [3]:
print("Loading raw CSVs...")
train_ops = pd.read_csv(TRAIN_OPS)
val_ops   = pd.read_csv(VAL_OPS)
test_ops  = pd.read_csv(TEST_OPS)
train_tte = pd.read_csv(TRAIN_TTE)
val_lbl   = pd.read_csv(VAL_LABELS)
print(f"Train ops shape : {train_ops.shape}")
print(f"Val ops shape   : {val_ops.shape}")
print(f"Test ops shape  : {test_ops.shape}")
print(f"Train TTE shape : {train_tte.shape}")
print(f"Val labels shape: {val_lbl.shape}")


Loading raw CSVs...
Train ops shape : (1122452, 107)
Val ops shape   : (119103, 107)
Test ops shape  : (275264, 107)
Train TTE shape : (23550, 3)
Val labels shape: (3027, 2)


In [4]:
# ── Raw Data Overview ────────────────────────────────────────────────────────
print("=" * 60)
print("  Scania APS Raw Data Overview")
print("=" * 60)

for split_name, df in [("Train", train_ops), ("Val", val_ops), ("Test", test_ops)]:
    vehicles = df["vehicle_id"].nunique()
    ts_span  = df.groupby("vehicle_id")["time_step"].agg(["min", "max"])
    ts_span["span"] = ts_span["max"] - ts_span["min"]
    print(f"\n── {split_name} ──")
    print(f"  Rows                  : {len(df):,}")
    print(f"  Columns               : {df.shape[1]}")
    print(f"  Unique vehicles       : {vehicles:,}")
    print(f"  time_step range       : {df['time_step'].min():.1f} ~ {df['time_step'].max():.1f}")
    print(f"  Avg time span/vehicle : {ts_span['span'].mean():.1f}")
    print(f"  Med time span/vehicle : {ts_span['span'].median():.1f}")
    print(f"  Min time span/vehicle : {ts_span['span'].min():.1f}")
    print(f"  Max time span/vehicle : {ts_span['span'].max():.1f}")

# ── Train: repaired vs non-repaired ──────────────────────────────────────────
# in_study_repair == 1: vehicle was repaired (APS fault) -> anomaly
# in_study_repair == 0: vehicle survived study period    -> censored/normal
print("\n── Train vehicle status (from train_tte) ──")
_rep_flag = (train_tte["in_study_repair"] >= 0.5).astype(int)
_n_normal = (_rep_flag == 0).sum()
_n_repair = (_rep_flag == 1).sum()
_total    = len(train_tte)
print(f"  Non-repaired (normal/censored): {_n_normal:,}  ({100*_n_normal/_total:.1f}%)")
print(f"  Repaired     (APS fault)      : {_n_repair:,}  ({100*_n_repair/_total:.1f}%)")
print(f"  Total vehicles                : {_total:,}")

# Row-level breakdown in train_ops
print("\n── Train row-level breakdown ──")
_tte_flag = train_tte[["vehicle_id", "in_study_repair"]].copy()
_tte_flag["in_study_repair"] = (_tte_flag["in_study_repair"] >= 0.5).astype(int)
_ops_flag = train_ops[["vehicle_id"]].merge(_tte_flag, on="vehicle_id", how="left")
_rows_normal = (_ops_flag["in_study_repair"] == 0).sum()
_rows_repair = (_ops_flag["in_study_repair"] == 1).sum()
_rows_total  = len(_ops_flag)
print(f"  Rows from non-repaired vehicles: {_rows_normal:,}  ({100*_rows_normal/_rows_total:.1f}%)")
print(f"  Rows from repaired vehicles    : {_rows_repair:,}  ({100*_rows_repair/_rows_total:.1f}%)")
print(f"  Total rows                     : {_rows_total:,}")

# ── Val label distribution (official, 5-class) ───────────────────────────────
_lbl_name = {
    0: "no fault      ",
    1: "fault class 1 ",
    2: "fault class 2 ",
    3: "fault class 3 ",
    4: "fault class 4 ",
}
print("\n── Val label distribution (official, 5-class) ──")
_vdist = val_lbl["class_label"].value_counts().sort_index()
for lbl, cnt in _vdist.items():
    name = _lbl_name.get(lbl, "")
    print(f"  label {lbl}  {name}: {cnt:>6,} vehicles  ({100*cnt/_vdist.sum():.1f}%)")
print(f"  Total vehicles: {_vdist.sum():,}")

# ── Test vehicles ────────────────────────────────────────────────────────────
print("\n── Test vehicles ──")
print(f"  Unique vehicles in test_ops: {test_ops['vehicle_id'].nunique():,}")

# ── Train label distribution (RUL-based, row level) ────────────────────────
# Rules (same as rul_to_ordinal in scania_feature_engineering.py):
#   non-repaired (in_study_repair==0) -> rul = inf -> label 0
#   rul = max(length_of_study_time_step - time_step, 0)
#   rul > 48 -> 0,  <=48 -> 1,  <=24 -> 2,  <=12 -> 3,  <=6 -> 4
print("\n── Train label distribution (RUL-based, row level) ──")
print("  Rules: non-repaired->0  |  rul>48->0, <=48->1, <=24->2, <=12->3, <=6->4")

_tte2 = train_tte[["vehicle_id", "length_of_study_time_step", "in_study_repair"]].copy()
_tte2["in_study_repair"] = (_tte2["in_study_repair"] >= 0.5).astype(int)
_tr2 = train_ops[["vehicle_id", "time_step"]].merge(_tte2, on="vehicle_id", how="left")
_rep_mask = _tr2["in_study_repair"] == 1
_tr2["rul"] = np.where(
    _rep_mask,
    np.maximum(_tr2["length_of_study_time_step"] - _tr2["time_step"], 0.0),
    np.inf,
)
_tr2["label"] = 0
_tr2.loc[np.isfinite(_tr2["rul"]) & (_tr2["rul"] <= 48), "label"] = 1
_tr2.loc[np.isfinite(_tr2["rul"]) & (_tr2["rul"] <= 24), "label"] = 2
_tr2.loc[np.isfinite(_tr2["rul"]) & (_tr2["rul"] <= 12), "label"] = 3
_tr2.loc[np.isfinite(_tr2["rul"]) & (_tr2["rul"] <=  6), "label"] = 4

_lbl_name2 = {
    0: "normal   (rul>48 or censored)",
    1: "risk-1   (24<rul<=48)        ",
    2: "risk-2   (12<rul<=24)        ",
    3: "risk-3   (6<rul<=12)         ",
    4: "imminent (rul<=6)            ",
}
_row_dist2 = _tr2["label"].value_counts().sort_index()
for lbl, cnt in _row_dist2.items():
    print(f"  label {lbl}  {_lbl_name2[lbl]}: {cnt:>10,} rows  ({100*cnt/len(_tr2):.2f}%)")
print(f"  Total rows: {len(_tr2):,}")

print("\n── Train label distribution (per vehicle -- worst label per vehicle) ──")
_veh_worst2 = _tr2.groupby("vehicle_id")["label"].max()
_veh_dist2  = _veh_worst2.value_counts().sort_index()
for lbl, cnt in _veh_dist2.items():
    print(f"  label {lbl}  {_lbl_name2[lbl]}: {cnt:>6,} vehicles  ({100*cnt/_veh_dist2.sum():.1f}%)")


  Scania APS Raw Data Overview

── Train ──
  Rows                  : 1,122,452
  Columns               : 107
  Unique vehicles       : 23,550
  time_step range       : 0.0 ~ 507.4
  Avg time span/vehicle : 227.2
  Med time span/vehicle : 208.4
  Min time span/vehicle : 11.2
  Max time span/vehicle : 504.2

── Val ──
  Rows                  : 119,103
  Columns               : 107
  Unique vehicles       : 3,027
  time_step range       : 0.0 ~ 445.4
  Avg time span/vehicle : 183.7
  Med time span/vehicle : 165.2
  Min time span/vehicle : 0.0
  Max time span/vehicle : 444.4

── Test ──
  Rows                  : 275,264
  Columns               : 107
  Unique vehicles       : 7,064
  time_step range       : 0.0 ~ 456.6
  Avg time span/vehicle : 184.2
  Med time span/vehicle : 163.8
  Min time span/vehicle : 0.0
  Max time span/vehicle : 449.6

── Train vehicle status (from train_tte) ──
  Non-repaired (normal/censored): 21,278  (90.4%)
  Repaired     (APS fault)      : 2,272  (9.6%)
  Tota